# 03 · Evaluación PropertyRAG (LlamaIndex)
Evalúa LlamaIndex PropertyGraphIndex (Gemini LLM + Ollama embeddings) sobre UltraDomain con métricas RAGAS.

In [1]:
import os
import sys
import nest_asyncio
from dotenv import load_dotenv

nest_asyncio.apply()
sys.path.append(os.path.abspath('../..'))
load_dotenv(os.path.join('../..', '.env'))

True

## 1. Configuración del experimento

In [2]:
DOMINIO     = "cs"
N_LIBROS    = None
MAX_Q       = None   # None = todas las preguntas
SHUFFLE     = False
RESULTS_DIR = "../../results_PropertyRAG/"

## 2. Cargar datos

In [3]:
from src.ultradomain import cargar_experimento

libros, qas = cargar_experimento(DOMINIO, n_libros=N_LIBROS, shuffle=SHUFFLE)

📚 Dominio: cs
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   Total Q&A: 100


## 3. Inicializar LlamaIndex

In [4]:
from llama_index.core import PropertyGraphIndex, Document, Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor
from src.limiters import RateLimitedGeminiEmbedding

if "GEMINI_API_KEY" in os.environ:
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

Settings.llm = Gemini(
    model="models/gemini-2.5-flash-lite",
    api_key=os.environ.get("GOOGLE_API_KEY"),
    max_tokens=2048,    # ← limita output para evitar MAX_TOKENS
)

from src.limiters import TruncatedOllamaEmbedding

Settings.embed_model = TruncatedOllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
)
Settings.embed_batch_size = 10

print("✅ LlamaIndex configurado")

/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/Volumes/Toni 1TB/MESIIA/TFM/TFM_Repositori/Code_TFM/venv/lib/python3.10/site-packages/llama_index/llms/gemini/base.py:21: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai
/var/folders/tf/rh2grj_j239fbyx5vlzz54r40000

✅ LlamaIndex configurado


## 4. Indexar documentos

In [5]:
from llama_index.core import PropertyGraphIndex, Document, Settings, StorageContext
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor
import os
import time
import sys
import logging
from llama_index.core.indices.property_graph import (
    VectorContextRetriever,
    LLMSynonymRetriever,
)

PERSIST_DIR = "../../propertyrag_storage"
storage_context = StorageContext.from_defaults()
documentos = [
    Document(text=libro["texto"], metadata={"titulo": libro["titulo"]})
    for libro in libros
]

index = PropertyGraphIndex.from_documents(documentos, show_progress=True)
elementos_extraidos = index.property_graph_store.get()
storage_context
query_engine = index.as_query_engine(include_text=True)



Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

Applying transformations:   0%|          | 0/2 [00:00<?, ?it/s]

Generating embeddings: 100%|██████████| 5881/5881 [16:18<00:00,  6.01it/s]


## 5. Ejecutar evaluación

In [6]:
from src.evaluation.experiment import run_experiment

result = await run_experiment(
    rag_type="llamaindex",
    rag_object=query_engine,
    dominio=DOMINIO,
    n_libros=N_LIBROS,
    max_questions=MAX_Q,
    shuffle=SHUFFLE,
    save_path=RESULTS_DIR,
)

📚 Dominio: cs
   📖 Mastering VBA for Microsoft Office 2013 — Richard Mansfield (15 preguntas)
   📖 Joe Celko's SQL Programming Style — Joe Celko (11 preguntas)
   📖 Introduction to the Theory of Programming Languages — Gilles Dowek (9 preguntas)
   📖 Guide to Java — James T. Streib (8 preguntas)
   📖 Professional Microsoft SQL Server 2008 Programming — Robert Vieira (11 preguntas)
   📖 Modern Optimization With R — Paulo Cortez (9 preguntas)
   📖 Linux Kernel Networking — Rami Rosen (8 preguntas)
   📖 Probability and Statistics for Computer Science — David Forsyth (9 preguntas)
   📖 Machine Learning With Spark — Nick Pentreath (8 preguntas)
   📖 Introducing Regular Expressions — Michael Fitzgerald (12 preguntas)
   Total Q&A: 100


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



🔍 Evaluando LLAMAINDEX | cs | 100 preguntas
────────────────────────────────────────────────────────────
  [1/100] How does Spark Streaming enable real-time data processing?...
  [2/100] What does the book suggest about the use of histograms in data analysi...
  [3/100] What are some advanced topics covered in the book related to Linux Ker...
  [4/100] What is the significance of the R tool in the context of modern optimi...
  [5/100] What are the key features of this text that aid in learning object-ori...
  [6/100] What is the role of the RegExr tool in the book?...
  [7/100] How does the text compare to other Java programming texts in terms of ...
  [8/100] What role do Bayesian inference and priors play in the book?...
  [9/100] What is the difference between recording a macro and writing code from...
  [10/100] How does the book address the implementation of IPv6 in comparison to ...
  [11/100] Can you explain the concept of standard coordinates as discussed in th...
  [12/100] W

## 6. Inspeccionar respuestas individuales

In [ ]:
for r in result.qa_results:
    print(f"❓ {r.question}")
    print(f"✅ GT:  {r.ground_truth[:150]}...")
    print(f"🤖 RAG: {r.answer[:150]}...")
    print(f"⏱️  {r.latency_s}s\n")